In [1]:

!uv add pip

Resolved 103 packages in 43ms
Checked 5 packages in 9ms


In [1]:
import os
import json

In [ ]:
json_file_path = "/path to your keys.json"
with open(json_file_path) as f:
    os.environ["DESTINATION__CREDENTIALS"] = f.read()
os.environ["BUCKET_URL"] = "gs://GCS_bucket_name"

In [4]:
import sys
!{sys.executable} -m pip install "dlt[bigquery,gs]" pandas pyarrow

In [5]:
# Install for testing
!pip install "dlt[duckdb]"

In [8]:
import dlt
import requests
import pandas as pd
from dlt.destinations import filesystem
from io import BytesIO

In [4]:
# Define a dlt source to download and process Parquet files as resources
@dlt.source(name="rides")
def download_parquet():
    prefix = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata"
    for month in range(6, 7):
        file_name = f"yellow_tripdata_2024-0{month}.parquet"
        url = f"{prefix}_2024-0{month}.parquet"
        response = requests.get(url)

        df = pd.read_parquet(BytesIO(response.content))

        # Return the dataframe as a dlt resource for ingestion
        yield dlt.resource(df, name=file_name)


# Initialize the pipeline
pipeline = dlt.pipeline(
    pipeline_name="rides_pipeline",
    destination=filesystem(layout="{schema_name}/{table_name}.{ext}"),
    dataset_name="rides_dataset",
)

# Run the pipeline to load Parquet data into DuckDB
load_info = pipeline.run(download_parquet(), loader_file_format="parquet")

# Print the results
print(load_info)


Pipeline rides_pipeline load step completed in 1.70 seconds
1 load package(s) were loaded to destination filesystem and into dataset rides_dataset
The filesystem destination used gs://kestra-zoom-demo-sravs location to store data
Load package 1780741811.338973 is LOADED and contains no failed jobs


In [ ]:
from dlt.destinations import bigquery
# Define a dlt resource to download and process Parquet files as single table
@dlt.resource(name="rides", write_disposition="replace")
def download_parquet():
    prefix = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata'

    for month in range(1, 7):
        url = f"{prefix}_2024-0{month}.parquet"
        response = requests.get(url)

        df = pd.read_parquet(BytesIO(response.content))

        yield df


# Initialize the pipeline
pipeline = dlt.pipeline(
    pipeline_name="rides_pipeline",
    #destination="duckdb",  # Use DuckDB for testing
    destination=bigquery(location="asia-south1"), # Use BigQuery for production
    dataset_name="rides_dataset",
)

# Run the pipeline to load Parquet data into DuckDB
info = pipeline.run(download_parquet)

# Print the results
print(info)

2026-06-06 11:18:37,596|[WARNING]|62264|128690616420160|dlt|pipeline.py|_state_to_props:1774|The destination dlt.destinations.duckdb:None in state differs from destination dlt.destinations.bigquery:bigquery in pipeline and will be ignored
/workspaces/Docker-workshop/module3_hw/.venv/lib/python3.13/site-packages/google/cloud/bigquery/client.py:613: UserWarning: Cannot create BigQuery Storage client, the dependency google-cloud-bigquery-storage is not installed.
  warnings.warn(


Pipeline rides_pipeline load step completed in 48.98 seconds
1 load package(s) were loaded to destination bigquery and into dataset rides_dataset
The bigquery destination used kestra-zoomcamp@kestrademo-497811.iam.gserviceaccount.com@kestrademo-497811 location to store data
Load package 1780744720.290642 is LOADED and contains no failed jobs


In [7]:
print(pipeline.pipeline_name)
print(os.getcwd())
print(os.listdir())

rides_pipeline
/workspaces/Docker-workshop/module3_hw
['.venv', 'main.py', 'gcp_creds.json', '.python-version', '.ipynb_checkpoints', 'uv.lock', 'README.md', 'Untitled.ipynb', 'pyproject.toml']


In [ ]:
import duckdb

conn = duckdb.connect(f"{pipeline.pipeline_name}.duckdb")

# Set search path to the dataset
conn.sql(f"SET search_path = '{pipeline.dataset_name}'")

# Describe the dataset to see loaded tables
res = conn.sql("DESCRIBE").df()
print(res)